# First Example - Tracer Breakthrough

In this lesson, you will:

- Create and connect your first system of unit operations.
- Run CADET and analyze the results.

## Example 1: Target Flow Sheet

We start with a simple system comprising three unit operations:
- An [Inlet](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.Inlet.html)
- A Column (modeled as a [LRM](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.LumpedRateModelWithoutPores.html))
- An [Outlet](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.Outlet.html)

```{figure} ./resources/flow_sheet_PFR.png
:width: 60%
```

We will load the column with a non-binding tracer molecule (such as e.g. Blue-Dextran) and inspect the shape and timing of the breakthrough curve.

## Setting Up the Model

The first step in to define a `ComponentSystem`.

#### Component System

- The `ComponentSystem` ensures consistency in the number of components across all modules (unit operations, binding models, events, etc.).
- Components can be named, which automatically adds legends to plots.

For advanced use, see the [Component System documentation](https://cadet-process.readthedocs.io/en/latest/reference/process_model/component_system.html).

## Unit Operations

A `UnitOperation` is a class that represents the physico-chemical behavior of an apparatus and holds its model parameters.
For an overview of all models in CADET-Process, refer to the [Documentation](https://cadet-process.readthedocs.io/en/v0.11.1/reference/processModel.html#unit-operation-models).

Each unit operation requires:
- A `ComponentSystem`
- A unique name (used later to reference the unit in the flow sheet for setting `Events` and `OptimizationVariables`).

### Inlet

The `Inlet` serves as the source for the system, creating arbitrary concentration profiles as boundary conditions. For more details, see the [Inlet documentation](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.Inlet.html).

- Concentration profiles are described using a third-degree piecewise polynomial for each component.
- Flow rate can be expressed as a third-degree piecewise polynomial.

Here, the flow rate is constant, so we set it directly on the object.

```{note}
CADET supports any consistent system of units, but we recommend using SI units.
```

Note that every unit operation model has different model parameters.

To display all parameters, simply print the `parameters` attribute.

### Outlet
The `Outlet` serves as the sink for the system. For more details, see the [Outlet documentation](https://cadet-process.readthedocs.io/en/v0.11.1/reference/generated/CADETProcess.processModel.Outlet.html).

Note that the `Outlet` unit does not have any model parameters.

### Column: Lumped Rate Model Without Pores

We use the `LumpedRateModelWithoutPores` for this example.
For model equations, see the [LRM documentation](https://cadet.github.io/master/modelling/unit_operations/lumped_rate_model_without_pores.html).
For parameters, refer to the [LRM parameters documentation](https://cadet.github.io/v5.1.0/interface/unit_operations/lumped_rate_model_without_pores.html#lumped-rate-model-without-pores-config).

Assume the following parameters:

| Parameter        | Value | Unit                       |
| ---------------- | ----- | -------------------------- |
| Length           | 10    | $\text{cm}$                |
| Diameter         | 2     | $\text{cm}$                |
| Porosity         | 0.4   | \-                         |
| Axial Dispersion | 1e-7  | $\text{m}^2 \text{s}^{-1}$ |

To instantiate, first import the unit operation class from the `processModel` module, then print the required parameters.

Let's fill in the missing parameters.

## Flow Sheet Connectivity

```{figure} ./resources/flow_sheet_PFR.png
:width: 60%
```

The `FlowSheet` manages the connectivity between unit operations.
For more details, see the [Flow Sheet documentation](https://cadet-process.readthedocs.io/en/latest/reference/process_model/flow_sheet.html).

Let's add the column analogously.

Now all unit operations are in the flow sheet, but we need to connect them.

Let's connect the column to the outlet as well.

## The Process

The `Process` class handles all time-dependent configurations.
Since we have no dynamic events, we only need to set the cycle time:

## Setting Up the Simulator and Running the Simulation

To simulate the process, configure a process simulator. If no path is specified, CADET-Process will auto-detect CADET.

To use a specific CADET version, specify the install path:

```python
process_simulator = Cadet(install_path='/path/to/cadet/executable')
```

To check that everything works correctly, you can call the check_cadet method.

Now, run the simulation.

## Plotting the Results

The `simulation_results` object contains the solution for every unit operation.

```{note}
By default, solutions for the inlet and outlet of all unit operations are stored. This is relevant for unit operations with transformative steps.
```

Solutions for all unit operations are stored as solution objects inside a nested dictionary, that can be accessed via dot notation.

From this the raw solution array can be also read out

or convenient in-built plotting methods can be used.

### Inlet Profile

### Outlet Profile

Equivalently, the outlet profile can be plot.

## Visualization

To visualize the column's internal state, enable the `SolutionRecorder` flag. The `SimulationResults` will then include an entry for the bulk.

```{note}
This solution is two-dimensional (space and time), so it can be plotted at a specific position (`plot_at_position`) or time (`plot_at_time`).
```

Use ipywidgets to visualize concentration propagation over the entire column for all time steps:

In [ ]:
%matplotlib widget

from ipywidgets import IntSlider, interactive_output
import ipywidgets as widgets
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


try:
    ui.close()
except NameError:
    pass
try:
    plt.close(fig)
except NameError:
    pass

# Fresh figure
fig, ax = plt.subplots(figsize=(10, 5))


def draw(time):
    ax.cla()
    ax.set_ylim(0, 1)
    simulation_results.solution.column.bulk.plot_at_time(time, ax=ax)
    fig.tight_layout()
    fig.canvas.draw_idle()


time_slider = IntSlider(
    min=0,
    max=int(process.cycle_time),
    step=25,
    layout={'width': '800px'},
    description='Time',
    style={'description_width': 'initial'},
)

ui = interactive_output(draw, {'time': time_slider})

display(time_slider, ui)
draw(0)